# Temporal steering with frozen ResNet50 spatial features

Run in order with the seeded recordings and pretrained backbone available. Each frame contributes a 9728-dimensional vector. All nine candidates use the same eligible target frames, initialization, minibatch order and validation criterion.

| Input | Visual frames (newest first) | Previous applied commands | Width |
| --- | --- | --- | ---: |
| A | t | — | 9728 |
| B | t, t−1, t−2 | — | 29184 |
| C | t, t−1, t−2 | t−1, t−2 | 29186 |

Each input is trained with raw labels, downweighted raw labels (indexed spike weight 0.1), and smoothed labels (indexed target). All large-turn multipliers are 1. Offline C scores condition on recorded past commands. Evaluate recovery separately in closed-loop driving.

In [ ]:
import json
from pathlib import Path
import numpy as np
from PIL import Image

from mydezero import cuda, no_grad, test_mode
import mydezero.functions as F
from mydezero.layers import Linear
from mydezero.models import ResNet50, VGG16


class SpatialSteeringResNet(ResNet50):
    """Extract spatial features from mean-centered BGR frames (N, 3, 120, 160)."""

    def __init__(self, pretrained=False):
        super().__init__(pretrained=pretrained)
        self.fc6 = Linear(1, in_size=512 * 3 * 5 + 2048)

    def frozen_features(self, images):
        with test_mode(), no_grad():
            x = F.pooling(F.relu(self.bn1(self.conv1(images))), 3, stride=2)
            mid = self.res3(self.res2(x))
            top = self.res5(self.res4(mid))
            local = mid.data.reshape(len(images), 512, 3, 5, 5, 4).mean(axis=(3, 5))
            global_features = top.data.mean(axis=(2, 3))
            xp = cuda.get_array_module(global_features)
            return xp.concatenate((local.reshape(len(images), -1), global_features), axis=1)


NOTEBOOK_NAME = "resnet50_temporal_spatial_9728"
WIDTH = {"A": 9728, "B": 29184, "C": 29186}

In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
import shutil
import json
from pathlib import Path
from uuid import uuid4

import numpy as np
from PIL import Image

from mydezero import cuda, no_grad, test_mode
import mydezero.functions as F
from mydezero.layers import Linear
from mydezero.optimizers import Adam
from donkeycar_app.dataset import preprocess_camera, read_split

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/mydezero").is_dir())
seed = 42
np.random.seed(seed)
use_gpu = cuda.gpu_enable and cuda.cupy.cuda.runtime.getDeviceCount() > 0
xp = cuda.cupy if use_gpu else np
run = NOTEBOOK_NAME + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
out = root / "runs/donkeycar" / run
weights_dir = root / "models/donkeycar" / run
manifest = root / "examples/donkeycar/road-seed-splits.json"
data_config = json.loads(manifest.read_text(encoding="utf-8"))
index_dir = root / data_config["index_dir"]
for folder in (out, weights_dir):
    folder.mkdir(parents=True)
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")

print(run, "CUDA" if use_gpu else "CPU")

In [ ]:
indices = {split: index_dir / f"{split}.json" for split in ("train", "validation", "test")}
rows = {split: read_split(path) for split, path in indices.items()}
seeds = {split: {r["seed"] for r in part} for split, part in rows.items()}
assert sum(map(len, seeds.values())) == len(set.union(*seeds.values()))
for split, part in rows.items():
    repaired = sum(r["raw"] != r["target"] for r in part)
    print(split, len(part), "frames,", repaired, "repaired labels")
    if split != "train":
        assert repaired == 0

## Consecutive windows and frozen features

Exclude the first two frames of each continuous route segment for every candidate. Cache one backbone output per frame. Assemble the temporal input only for the current minibatch.

In [ ]:
def endpoints(part):
    return np.asarray([i for i in range(2, len(part))
                       if all(part[i-k]["route"] == part[i]["route"] and
                              part[i-k]["frame"] == part[i]["frame"] - k for k in (1, 2))],
                      dtype=np.int64)

ends = {s: endpoints(part) for s, part in rows.items()}
eligible = {s: [rows[s][i] for i in ids] for s, ids in ends.items()}
assert all(len(ids) for ids in ends.values())
for split, ids in ends.items():
    print(split, len(ids), "eligible of", len(rows[split]))

In [ ]:
model = SpatialSteeringResNet(pretrained=True)
if use_gpu:
    model.to_gpu()
def pixels(row):
    with Image.open(root / row["image"]) as im:
        if im.mode != "RGB" or im.size != (160, 120):
            raise ValueError(row["image"])
        return preprocess_camera(im)

def features(part):
    result = np.empty((len(part), 9728), dtype=np.float32)
    for start in range(0, len(part), 32):
        images = np.stack([pixels(r) for r in part[start:start + 32]])
        values = model.frozen_features(xp.asarray(images))
        result[start:start + len(images)] = cuda.as_numpy(values)
    return result

visual = {s: features(rows[s]) for s in ("train", "validation")}
center = visual["train"].mean(axis=0)
scale = np.maximum(visual["train"].std(axis=0), 0.1)
def standardize(values):
    values -= center
    values /= scale

for values in visual.values():
    standardize(values)
commands = {s: np.asarray([r["raw"] for r in part], dtype=np.float32) for s, part in rows.items()}

def inputs(split, mode, positions):
    ids = ends[split][positions]
    frames = visual[split]
    if mode == "A":
        return xp.asarray(frames[ids])
    blocks = [frames[ids-k] for k in (0, 1, 2)]
    if mode == "C":
        blocks.extend(commands[split][ids-k, None] for k in (1, 2))
    return xp.asarray(np.concatenate(blocks, axis=1))

## Fit and select nine readouts

Use the original smooth absolute-error loss and Adam schedule. Clip predictions for evaluation. Fold the visual standardization into each exported readout so a checkpoint accepts unstandardized spatial features.

In [ ]:
epochs, batch_size, delta = 24, 256, 0.05
recipes = ("raw", "downweighted_x1", "smoothed_x1")
fit = eligible["train"]
def predict(split, mode, weight, bias):
    batches = []
    for start in range(0, len(ends[split]), batch_size):
        x = inputs(split, mode, slice(start, start + batch_size))
        batches.append(cuda.as_numpy(xp.clip(x @ weight + bias, -1, 1)).ravel())
    return np.concatenate(batches)

def train(mode, recipe):
    head = Linear(1, in_size=WIDTH[mode])
    head.W.data.fill(0)
    if use_gpu:
        head.to_gpu()
    optim = Adam(lr=0.00008).setup(head)
    label = "target" if recipe == "smoothed_x1" else "raw"
    y = xp.asarray([r[label] for r in fit], dtype=xp.float32)[:, None]
    w = xp.asarray([r["weight"] if recipe == "downweighted_x1" else 1 for r in fit],
                   dtype=xp.float32)[:, None]
    target = commands["validation"][ends["validation"]]
    rng = np.random.default_rng(seed)
    curve = []
    for epoch in range(1, epochs + 1):
        optim.lr = 0.00008 * 0.5 ** ((epoch - 1) // 8)
        order = rng.permutation(len(fit))
        for start in range(0, len(order), batch_size):
            ids = order[start:start + batch_size]
            residual = head(inputs("train", mode, ids)) - y[ids]
            loss = F.sum(((residual ** 2 + delta ** 2) ** 0.5 - delta) * w[ids]) / float(w[ids].sum())
            head.cleargrads()
            loss.backward()
            optim.update()
        pred = predict("validation", mode, head.W.data, head.b.data)
        mae = float(np.abs(pred - target).mean())
        if not np.isfinite(mae):
            raise ValueError(f"Non-finite validation MAE: {mode}/{recipe}, epoch {epoch}")
        if not curve or mae < min(curve):
            best = (epoch, head.W.data.copy(), head.b.data.copy())
        curve.append(mae)
    return best, curve

def export_head(mode, weight, bias, path):
    repeats = 1 if mode == "A" else 3
    visual_width = repeats * 9728
    folded = weight.copy()
    folded[:visual_width] /= xp.tile(xp.asarray(scale), repeats)[:, None]
    model.fc6.W.data = folded
    model.fc6.b.data = bias - xp.tile(xp.asarray(center), repeats) @ folded[:visual_width]
    model.save_weights(path)

heads, candidates = {}, {}
for mode in WIDTH:
    for recipe in recipes:
        name = f"{mode}-{recipe}"
        (epoch, weight, bias), curve = train(mode, recipe)
        heads[name] = (weight, bias)
        export_head(mode, weight, bias, weights_dir / f"{name}.npz")
        candidates[name] = dict(mode=mode, recipe=recipe, epoch=epoch, validation_mae=min(curve),
                                validation_curve=curve)
        print(name, epoch, min(curve), flush=True)
winner = min(candidates, key=lambda name: candidates[name]["validation_mae"])
checkpoint = weights_dir / "best.npz"
shutil.copyfile(weights_dir / f"{winner}.npz", checkpoint)
print("Selected", winner, checkpoint)

## Verify the exported winner and evaluate every candidate

Compare a reloaded CPU checkpoint on original RGB windows with cached validation predictions. Report test MAE for each route, large turns, sustained turns and spikes. The seeded test routes are a historical holdout. Test scores do not affect selection.

In [ ]:
mode = candidates[winner]["mode"]
loaded = SpatialSteeringResNet()
loaded.load_weights(checkpoint)
assert loaded.fc6.W.shape == (WIDTH[mode], 1)
expected = predict("validation", mode, *heads[winner])[:4]
actual = []
for i in ends["validation"][:4]:
    lags = (0,) if mode == "A" else (0, 1, 2)
    frames = np.stack([pixels(rows["validation"][i-k]) for k in lags])
    vector = loaded.frozen_features(frames).reshape(1, -1)
    if mode == "C":
        vector = np.concatenate((vector, commands["validation"][[i-1, i-2]][None]), axis=1)
    actual.append(float(np.clip(vector @ loaded.fc6.W.data + loaded.fc6.b.data, -1, 1).item()))
np.testing.assert_allclose(actual, expected, atol=3e-5, rtol=1e-4)
print("CPU checkpoint round trip passed")

In [ ]:
def scores(part, predictions):
    targets = np.array([r["raw"] for r in part])
    err = np.abs(predictions - targets)
    zero = np.abs(targets)
    groups = {"all": np.ones(len(part), dtype=bool),
              "sharp": zero >= 0.3,
              "sustained": np.array([r["sustained"] for r in part]),
              "spikes": np.array([r["spike"] for r in part]),
              "sharp_nonspike": (zero >= 0.3) & ~np.array([r["spike"] for r in part]),
              "ordinary": zero < 0.3}
    for route in sorted({r["route"] for r in part}):
        groups[route] = np.array([r["route"] == route for r in part])
    return {k: dict(frames=int(mask.sum()), mae=float(err[mask].mean()),
                    zero_mae=float(zero[mask].mean())) for k, mask in groups.items() if mask.any()}

if use_gpu:
    model.to_gpu()  # Saving weights moves MyDeZero layers to CPU.
visual["test"] = features(rows["test"])
standardize(visual["test"])
report = dict(run=run, seed=seed, selected=winner, checkpoint=checkpoint.relative_to(root).as_posix(),
              selection="minimum overall validation MAE", epochs=epochs, batch_size=batch_size,
              index_sha256={s: hashlib.sha256(path.read_bytes()).hexdigest() for s, path in indices.items()},
              candidates=candidates)
for split in ("validation", "test"):
    part = eligible[split]
    predictions = {}
    for name, params in heads.items():
        pred = predict(split, candidates[name]["mode"], *params)
        assert np.isfinite(pred).all()
        predictions[name] = pred
        candidates[name][split] = scores(part, pred)
    with (out / f"{split}-predictions.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["route", "frame", "raw", *predictions])
        for i, r in enumerate(part):
            writer.writerow([r["route"], r["frame"], r["raw"], *(p[i] for p in predictions.values())])
save_json(out / "metrics.json", report)
print("Candidate               Epoch    Val MAE   Test MAE")
for name, info in candidates.items():
    print(f"{name:22s} {info['epoch']:5d} {info['validation_mae']:10.5f} {info['test']['all']['mae']:10.5f}")
print("Best by validation:", winner, checkpoint)

## Drive a saved checkpoint (optional)

Run the imports and model definition, then supply a checkpoint. Use one fresh controller per episode. The simulator helper calls the controller once on a dummy image before reset. Discard that warm-up. At episode start repeat the earliest observation and use zero for missing commands. With lane assistance disabled the clipped prediction is the applied steering.

In [ ]:
def drive_saved(path):
    from donkeycar_app.drive import run_drive
    net = SpatialSteeringResNet()
    net.load_weights(path)
    mode = next(m for m, width in WIDTH.items() if width == net.fc6.W.shape[0])
    device = cuda.cupy if cuda.gpu_enable and cuda.cupy.is_available() else np
    if device is not np:
        net.to_gpu()
    previous_frames = []
    past_actions = [0.0, 0.0]
    warmup = True
    def steer(frame):
        nonlocal warmup
        pixels = VGG16.preprocess(Image.fromarray(frame), size=None)
        feat = net.frozen_features(device.asarray(np.ascontiguousarray(pixels[None])))
        if warmup:
            warmup = False
            return 0.0
        if not previous_frames:
            previous_frames.extend([feat, feat])
        blocks = [feat] if mode == "A" else [feat, *previous_frames]
        if mode == "C":
            blocks.append(device.asarray([past_actions], dtype=device.float32))
        vector = device.concatenate(blocks, axis=1)
        action = float(device.clip(vector @ net.fc6.W.data + net.fc6.b.data, -1, 1).item())
        previous_frames[:] = [feat, previous_frames[0]]
        past_actions[:] = [action, past_actions[0]]
        return action
    run_drive(steer, lane_assist=False, log_prefix=f"temporal-{mode}-{Path(path).stem}")

# drive_saved(checkpoint)